# MedSAM segmentation — Oral cancer histopathology

Runs `auto_segment.py` from [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) on a Kaggle GPU.

For every image MedSAM gets a **label-free box prompt** (the extent of the stained tissue), predicts a mask, and the mask's bounding box is cropped as the ROI.

### Before running
1. **Upload the data as a private Kaggle Dataset** containing these two folders from the project:
   - `preprocessed_normalized/`
   - `splits/`

   Keep it **private**: the Normal images come from a CC BY-NC-ND dataset, which does not allow sharing modified versions.
2. In this notebook: **Add Input** → select that dataset.
3. **Settings** → Accelerator: **GPU T4** (or P100), **Internet: On** (to clone the code and download MedSAM).

### How to run
- Run the cells up to **Quick test** and look at the visualizations.
- If they look reasonable, use **Save Version → Save & Run All** for the full run. It keeps running in the background and the outputs are saved with the version.

## 1. Check the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

## 2. Get the code

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)

print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Find the uploaded data

In [ ]:
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("splits/normalized_dataset.csv"))
assert matches, "Could not find splits/normalized_dataset.csv under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent
OUTPUT_ROOT = Path("/kaggle/working")

assert (DATA_ROOT / "preprocessed_normalized").exists(), f"preprocessed_normalized/ not found next to splits/ in {DATA_ROOT}"

os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

# Reduces GPU memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

## 4. Quick test (3 images per class)

Check these before the full run. Things to look for:
- Does the mask cover meaningful tissue (e.g. epithelium), or just an arbitrary blob?
- Is it different between classes, or the same shape every time?
- `MedSAM predicted IoU` is the model's own confidence, not an accuracy measure.

In [ ]:
!cd {CODE_DIR} && python auto_segment.py --per-class 3 --vis-per-class 3

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

vis = sorted((OUTPUT_ROOT / "segmentation" / "visualizations").rglob("*.png"))
print(len(vis), "visualizations")

for path in vis:
    plt.figure(figsize=(15, 5))
    plt.imshow(Image.open(path))
    plt.axis("off")
    plt.show()

## 5. Full run

Segments all images (train, validation, test). Uses `--resume`, so if the session stops you can simply run this cell again.

The quick-test results are kept and skipped.

In [ ]:
!cd {CODE_DIR} && python auto_segment.py --resume

## 6. Summary

In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT_ROOT / "segmentation" / "segmentation_results.csv")
print("Images:", len(results))
print()
print(results.groupby(["split", "class"]).size().unstack(0))
print()
print(results.groupby("class")[["mask_fraction", "medsam_iou_score", "roi_width", "roi_height"]].describe().T.round(3))
print()
print(results["roi_status"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for cls, g in results.groupby("class"):
    axes[0].hist(g["mask_fraction"], bins=40, alpha=0.5, label=cls)
    axes[1].hist(g["medsam_iou_score"], bins=40, alpha=0.5, label=cls)
axes[0].set_title("Mask area (fraction of image)")
axes[1].set_title("MedSAM predicted IoU")
axes[0].legend()
plt.show()

## 7. Package results for download

Creates `segmentation.zip` in the output panel (masks, ROIs, visualizations and the results CSV). Paths in the CSV are relative, so after unzipping into the project folder on your laptop they work directly.

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_ROOT / "segmentation"), "zip", OUTPUT_ROOT, "segmentation")
print(archive, round(os.path.getsize(archive) / 2**30, 2), "GB")